# Logic converted from matlab file

In [1]:
import numpy as np
import nibabel as nib
import scipy.io as sio
import os
import h5py
from scipy.stats import zscore
from sklearn.linear_model import LinearRegression
from joblib import Parallel, delayed


# --------------------------------------------------
# Utility: Compute R² between residual and original
# --------------------------------------------------

def rsquared(Xresid, Xorig):
    return 1 - np.sum((Xresid) ** 2) / np.sum((Xorig - np.mean(Xorig)) ** 2)


# ----------------------------------------------------------------------------
# Main Function: Perform voxel-wise regression of PRF features on NSD betas
# isub: subject index (1-based like MATLAB)
# visual_regions: list of visual regions (e.g., [1, 2, 3])
# to_zscore: normalization method (0: none, 1: z-score, 2: mean, etc.)
# DEBUG_MODE (bool): If True, run on small subset of sessions, voxels, and trials
# ----------------------------------------------------------------------------


def regress_prf_split_expand(isub, visual_regions=[1], to_zscore=0, DEBUG_MODE=False):
    # -------------------------------
    # Setup & File Paths
    # -------------------------------

    # Number of sessions per subject
    nsessions_sub = [40, 40, 32, 30, 40, 32, 40, 30]
    nsessions = nsessions_sub[isub - 1]
    nsplits = nsessions  # One split per session

    # If debugging: reduce sessions
    if DEBUG_MODE:
        nsessions = min(nsessions, 2)
        nsplits = nsessions

    # bandpass info
    bandpass = 1
    band_min = 1
    band_max = 7

    # Set data folders
    prf_folder = r"C:\nsd\prfsample_expand"
    save_folder = r"C:\nsd\repDrift_expand/"
    nsd_folder = r"D:\NSD"
    betas_folder = os.path.join(nsd_folder, f"subj{isub:02d}", "func1pt8mm")
    visual_rois_file = os.path.join(betas_folder, "roi", "prf-visualrois.nii.gz")

    # Load visual ROI mask (NIfTI)
    vis_roi_img = nib.load(visual_rois_file)
    vis_roi_data = vis_roi_img.get_fdata().flatten()  # Flatten for voxel indexing

    roi_names = ["V1v", "V1d", "V2v", "V2d", "V3v", "V3d", "hV4"]

    # ------------------------------------------------------------
    # Define visual region mapping (visualRegion → ROI indices)
    # ------------------------------------------------------------

    # Each visualRegion value corresponds to a group of ROIs:
    #   1 → [V1v, V1d] (ROI indices 0 and 1)
    #   2 → [V2v, V2d] (ROI indices 2 and 3)
    #   3 → [V3v, V3d] (ROI indices 4 and 5)
    #   4 → [hV4]      (ROI index 6)
    roi_map = {
        1: [0, 1],  # V1v + V1d
        2: [2, 3],  # V2v + V2d
        3: [4, 5],  # V3v + V3d
        4: [6],     # hV4
    }

    # Load NSD experiment design
    nsd_design = sio.loadmat(os.path.join(nsd_folder, "nsd_expdesign.mat"))
    master_ordering = nsd_design["masterordering"].squeeze() - 1  # MATLAB 1-indexed
    sub_design = nsd_design["subjectim"][isub - 1, master_ordering].squeeze()

    # ------------------------------------------
    # Split trial matrix (1 row per split)
    # ------------------------------------------

    ntrials = len(master_ordering)
    trials_per_session = 10 * 75  # Each session: 10 blocks × 75 trials
    split_img_trials = np.zeros((nsplits, ntrials), dtype=bool)

    itrial = 0
    for isplit in range(nsplits):  # Define split_img_trials once
        split_img_trials[isplit, itrial : itrial + trials_per_session] = True
        itrial += trials_per_session

    # ------------------------------------------
    # Loop over Visual Regions
    # ------------------------------------------

    for visual_region in visual_regions: # # Outer loop: per visual ROI (e.g., V1v, V1d...)
        rois = roi_map.get(visual_region, [])
        print(f"Processing visual region: {visual_region}")

        # --- Load PRF sample features for this region ---
        prf_file = os.path.join(
            prf_folder,
            f"prfSampleStim_v{visual_region}_sub{isub}.h5",  # Load the PRF energy for this region
        )
        with h5py.File(prf_file, "r") as f:
            # Load scalar attributes
            num_levels = int(f.attrs["numLevels"])
            num_orientations = int(f.attrs["numOrientations"])
            roi_prf = None  # Optional — only used for saving, not needed for regression

            # Load global image list and ROI list
            all_imgs = np.array(f["allImgs"]).squeeze()
            rois = np.array(f["rois"]).squeeze()

            # Load prfSampleLev and prfSampleLevOri into Python dictionaries
            prf_sample_lev = {}
            prf_sample_lev_ori = {}

            for roinum in rois:
                prf_sample_lev[roinum] = np.array(f[f"prfSampleLev/roi_{roinum}"])  # shape: (n_imgs, n_vox, num_levels + 2)
                prf_sample_lev_ori[roinum] = np.array(f[f"prfSampleLevOri/roi_{roinum}"])  # shape: (n_imgs, n_vox, num_levels + 2, num_orientations)

        # --- Init containers for each ROI ---
        roi_betas = {roinum: [] for roinum in range(len(rois))}  # beta responses
        roi_ind = {}  # voxel indices per ROI

        # --- Load and normalize beta data per session ---
        for isession in range(1, nsessions + 1):  # For each session
            beta_file = os.path.join(betas_folder, f"betas_session{isession:02d}.nii")
            beta_img = nib.load(beta_file)
            beta_data = beta_img.get_fdata().astype(np.float64)
            beta_data = beta_data / 300  # Convert to percent signal change
            beta_data = beta_data.reshape(-1, beta_data.shape[-1])  # Flatten 4D to 2D

            for roinum, iroi in enumerate(rois): # For each ROI within this visual region
                vox_idx = np.where(vis_roi_data == iroi)[0]
                if DEBUG_MODE:
                    vox_idx = vox_idx[:10]  # Only first 10 voxels

                roi_ind[roinum] = vox_idx
                roi_voxels = beta_data[vox_idx, :]  # Get betas for this ROI

                # --- Apply normalization method ---
                if to_zscore == 1:
                    norm_voxels = zscore(roi_voxels, axis=1, ddof=0)
                elif to_zscore == 2:
                    norm_voxels = roi_voxels - roi_voxels.mean(axis=1, keepdims=True)
                elif to_zscore == 3:
                    mean_ = roi_voxels.mean(axis=1, keepdims=True)
                    norm_voxels = mean_ + zscore(roi_voxels - mean_, axis=1, ddof=0)
                elif to_zscore == 4:
                    norm_voxels = roi_voxels - roi_voxels.mean()
                else:
                    norm_voxels = roi_voxels  # No normalization

                roi_betas[roinum].append(norm_voxels)

        # --- Concatenate sessions along time axis ---
        for roinum in roi_betas:
            roi_betas[roinum] = np.concatenate(roi_betas[roinum], axis=1)

        # --- Match presented images to full image set ---
        img_trials_mask, img_num = (
            np.isin(sub_design, all_imgs, assume_unique=True),
            None,
        )
        img_num = np.searchsorted(
            all_imgs, sub_design[img_trials_mask]
        )  # Shape: (n_trials,)

        # Debug mode: reduce number of trials
        if DEBUG_MODE:
            for roinum in roi_betas:
                roi_betas[roinum] = roi_betas[roinum][:, :500]
            split_img_trials = split_img_trials[:, :500]
            img_num = img_num[:500]

        # --- Ensure trial counts match for this subject ---
        for roinum in roi_betas:
            roi_betas[roinum] = roi_betas[roinum][:, : split_img_trials.shape[1]]
        split_img_trials = split_img_trials[:, : roi_betas[roinum].shape[1]]
        img_num = img_num[: roi_betas[roinum].shape[1]]

        max_num_trials = split_img_trials.sum(axis=1).max()  # Used for array size later

        # Prepare containers per ROI
        nsd = {}
        nvox = {}
        r2 = {}
        r2ori = {}
        r2split = {}
        r2ori_split = {}
        rss_split = {}
        rss_ori_split = {}
        pearson_r = {}
        pearson_rori = {}
        sess_betas = {}
        sess_std_betas = {}

        vox_coef = {}
        vox_ori_coef = {}
        vox_pred_ori_coef = {}
        vox_ori_pred_ori_coef = {}
        vox_resid_ori_coef = {}
        vox_ori_resid_ori_coef = {}

        vox_pred_ori_r2 = {}
        vox_resid_ori_r2 = {}
        vox_ori_pred_ori_r2 = {}
        vox_ori_resid_ori_r2 = {}

        vox_coef_corr = {}
        vox_ori_coef_corr = {}
        vox_coef_corr_with_const = {}
        vox_ori_coef_corr_with_const = {}

        # ------------------------------------------------------------
        # Voxel-wise Regression for Each ROI and Each Split
        # ------------------------------------------------------------
        for roinum in range(len(rois)):
            print(f"  → Processing ROI {roinum} (ID {rois[roinum]})")
            nvox[roinum] = roi_betas[roinum].shape[0]
            L = nvox[roinum]

            # Allocate arrays for this ROI
            shape_lev = num_levels + 1
            shape_ori = num_levels * num_orientations + 1

            # Initialize coefficient and residual matrices per voxel per split
            vox_coef[roinum] = np.zeros((nsplits, L, shape_lev))
            vox_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))
            vox_pred_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))
            vox_ori_pred_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))
            vox_resid_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))
            vox_ori_resid_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))

            vox_pred_ori_r2[roinum] = np.zeros((nsplits, L))
            vox_resid_ori_r2[roinum] = np.zeros((nsplits, L))
            vox_ori_pred_ori_r2[roinum] = np.zeros((nsplits, L))
            vox_ori_resid_ori_r2[roinum] = np.zeros((nsplits, L))
            r2[roinum] = np.zeros((nsplits, L))
            r2ori[roinum] = np.zeros((nsplits, L))
            sess_betas[roinum] = np.zeros((nsplits, L))
            sess_std_betas[roinum] = np.zeros((nsplits, L))

            # Loop over splits
            for isplit in range(nsplits):
                img_trials_split = split_img_trials[isplit]
                trial_idx = np.where(img_trials_split)[0]

                betas_split = roi_betas[roinum][:, trial_idx]  # shape: voxels × trials
                sess_betas[roinum][isplit] = np.mean(betas_split, axis=1)
                sess_std_betas[roinum][isplit] = np.std(betas_split, axis=1)

                for ivox in range(L):
                    # Voxel betas
                    y = betas_split[ivox, :]

                    # ---------- PRF: Basic Level Model ----------
                    X = prf_sample_lev[roinum][img_num[trial_idx], :, :]  # shape: (n_trials, n_vox, num_levels+2)
                    X_vox = X[:, ivox, :]  # shape: (n_trials, num_levels+2)
                    X = np.column_stack(
                        [X_vox, np.ones(X_vox.shape[0])]
                    )  # Add constant term

                    coef = np.linalg.lstsq(X, y, rcond=None)[0]
                    vox_coef[roinum][isplit, ivox, :] = coef
                    pred = np.dot(X, coef) # pred = X @ coef
                    residual = y - pred
                    r2[roinum][isplit, ivox] = rsquared(residual, y)

                    # ---------- PRF: Orientation-Level Model ----------
                    X_ori_all = prf_sample_lev_ori[roinum][img_num[trial_idx], :, :, :]  # (n_trials, n_vox, lev, ori)
                    X_ori_vox = X_ori_all[:, ivox, :, :].reshape(len(trial_idx), -1)     # → (n_trials, lev * ori)
                    X_ori = np.column_stack(
                        [
                            X_ori_vox,
                            X_vox[:, -2],  # lowest SF
                            X_vox[:, -3],  # highest SF
                            np.ones(X_vox.shape[0]),
                        ]
                    )

                    coef_ori = np.linalg.lstsq(X_ori, y, rcond=None)[0]
                    vox_ori_coef[roinum][isplit, ivox, :] = coef_ori
                    pred_ori = np.dot(X_ori, coef_ori)  # pred_ori = X_ori @ coef_ori
                    residual_ori = y - pred_ori
                    r2ori[roinum][isplit, ivox] = rsquared(residual_ori, y)

                    # ---------- Residual regressions ----------
                    # Predict orientation residuals with orientation model
                    vox_ori_resid_ori_coef[roinum][isplit, ivox, :] = np.linalg.lstsq(
                        X_ori, residual_ori, rcond=None
                    )[0]
                    pred_ori_resid = np.dot(X_ori,vox_ori_resid_ori_coef[roinum][isplit, ivox, :])  # pred_ori_resid = X_ori @ vox_ori_resid_ori_coef[roinum][isplit, ivox, :]
                    vox_ori_resid_ori_r2[roinum][isplit, ivox] = rsquared(
                        residual_ori - pred_ori_resid, y
                    )

                    # Predict basic residuals with orientation model
                    vox_resid_ori_coef[roinum][isplit, ivox, :] = np.linalg.lstsq(
                        X_ori, residual, rcond=None
                    )[0]
                    pred_resid_ori =np.dot(X_ori, vox_resid_ori_coef[roinum][isplit, ivox, :]) # pred_resid_ori = X_ori @ vox_resid_ori_coef[roinum][isplit, ivox, :]
                    vox_resid_ori_r2[roinum][isplit, ivox] = rsquared(
                        residual - pred_resid_ori, y
                    )

                    # Predict prediction with orientation model
                    pred_from_basic = pred
                    vox_pred_ori_coef[roinum][isplit, ivox, :] = np.linalg.lstsq(
                        X_ori, pred_from_basic, rcond=None
                    )[0]
                    pred_pred_ori = np.dot(X_ori, vox_pred_ori_coef[roinum][isplit, ivox, :])
                    vox_pred_ori_r2[roinum][isplit, ivox] = rsquared(
                        pred_from_basic - pred_pred_ori, y
                    )

                    # Predict full orientation prediction with same
                    vox_ori_pred_ori_coef[roinum][isplit, ivox, :] = np.linalg.lstsq(
                        X_ori, pred_ori, rcond=None
                    )[0]
                    pred_ori_pred_ori = (
                        np.dot(X_ori, vox_ori_pred_ori_coef[roinum][isplit, ivox, :])
                    )
                    vox_ori_pred_ori_r2[roinum][isplit, ivox] = rsquared(
                        pred_ori - pred_ori_pred_ori, y
                    )
            # Containers for cross-split comparisons
            r2split[roinum] = np.zeros((nsplits, L, nsplits))
            r2ori_split[roinum] = np.zeros((nsplits, L, nsplits))
            rss_split[roinum] = np.zeros((nsplits, L, nsplits))
            rss_ori_split[roinum] = np.zeros((nsplits, L, nsplits))
            pearson_r[roinum] = np.zeros((nsplits, L, nsplits))
            pearson_rori[roinum] = np.zeros((nsplits, L, nsplits))

            # ----------------------------
            # Loop: isplit (target session)
            # ----------------------------
            for isplit in range(nsplits):
                trial_idx = np.where(split_img_trials[isplit])[0]
                y = roi_betas[roinum][:, trial_idx]  # shape: (voxels, trials)

                for ivox in range(L):
                    y_vox = y[ivox, :]

                    # Get design matrices for isplit
                    X = prf_sample_lev[roinum][img_num[trial_idx], :, :]  # shape: (n_trials, n_vox, num_levels+2)
                    X_vox = X[:, ivox, :]  # shape: (n_trials, num_levels+2)
                    X = np.column_stack([X_vox, np.ones(X_vox.shape[0])])

                    X_ori_all = prf_sample_lev_ori[roinum][img_num[trial_idx], :, :, :]  # (n_trials, n_vox, lev, ori)
                    X_ori_vox = X_ori_all[:, ivox, :, :].reshape(len(trial_idx), -1)     # → (n_trials, lev * ori)
                    X_ori = np.column_stack(
                        [
                            X_ori_vox,
                            X_vox[:, -2],  # lowest SF
                            X_vox[:, -3],  # highest SF
                            np.ones(X_vox.shape[0]),
                        ]
                    )

                    # ----------------------------
                    # Loop: nextSplit (model source)
                    # ----------------------------
                    for next_split in range(nsplits):
                        coef = vox_coef[roinum][next_split, ivox, :]
                        coef_ori = vox_ori_coef[roinum][next_split, ivox, :]

                        # Predict from basic model
                        y_pred = np.dot(X, coef)  # y_pred = X @ coef
                        residual = y_vox - y_pred
                        r2split[roinum][isplit, ivox, next_split] = rsquared(
                            residual, y_vox
                        )
                        rss_split[roinum][isplit, ivox, next_split] = np.sum(
                            residual**2
                        )
                        pearson_r[roinum][isplit, ivox, next_split] = np.corrcoef(
                            y_vox, y_pred
                        )[0, 1]

                        # Predict from orientation model
                        y_pred_ori =np.dot(X_ori, coef_ori)  # y_pred_ori = X_ori @ coef_ori
                        residual_ori = y_vox - y_pred_ori
                        r2ori_split[roinum][isplit, ivox, next_split] = rsquared(
                            residual_ori, y_vox
                        )
                        rss_ori_split[roinum][isplit, ivox, next_split] = np.sum(
                            residual_ori**2
                        )
                        pearson_rori[roinum][isplit, ivox, next_split] = np.corrcoef(
                            y_vox, y_pred_ori
                        )[0, 1]

            # --------------------------------------------------------
            # Correlation Between Splits for Each Voxel
            # --------------------------------------------------------
            vox_coef_corr[roinum] = np.zeros((L, nsplits, nsplits))
            vox_ori_coef_corr[roinum] = np.zeros((L, nsplits, nsplits))
            vox_coef_corr_with_const[roinum] = np.zeros((L, nsplits, nsplits))
            vox_ori_coef_corr_with_const[roinum] = np.zeros((L, nsplits, nsplits))

            for ivox in range(L):
                # Without constant (drop last column)
                coef_mat = vox_coef[roinum][:, ivox, :-1]
                ori_mat = vox_ori_coef[roinum][:, ivox, :-1]

                vox_coef_corr[roinum][ivox] = np.corrcoef(coef_mat, rowvar=False)
                vox_ori_coef_corr[roinum][ivox] = np.corrcoef(ori_mat, rowvar=False)

                # With constant
                coef_mat_full = vox_coef[roinum][:, ivox, :]
                ori_mat_full = vox_ori_coef[roinum][:, ivox, :]

                vox_coef_corr_with_const[roinum][ivox] = np.corrcoef(
                    coef_mat_full, rowvar=False
                )
                vox_ori_coef_corr_with_const[roinum][ivox] = np.corrcoef(
                    ori_mat_full, rowvar=False
                )

        # -----------------------------------------------------
        # Package Results in Dictionary and Save
        # -----------------------------------------------------
        nsd = {
            "voxOriCoefCorrWithConst": vox_ori_coef_corr_with_const,
            "voxCoefCorrWithConst": vox_coef_corr_with_const,
            "voxOriCoefCorr": vox_ori_coef_corr,
            "voxCoefCorr": vox_coef_corr,
            "r2": r2,
            "r2ori": r2ori,
            "r2split": r2split,
            "r2oriSplit": r2ori_split,
            "rssOriSplit": rss_ori_split,
            "rssSplit": rss_split,
            "pearsonRori": pearson_rori,
            "pearsonR": pearson_r,
            "imgNum": img_num,
            "splitImgTrials": split_img_trials,
            "voxCoef": vox_coef,
            "voxOriCoef": vox_ori_coef,
            "voxPredOriCoef": vox_pred_ori_coef,
            "voxOriPredOriCoef": vox_ori_pred_ori_coef,
            "voxResidOriCoef": vox_resid_ori_coef,
            "voxOriResidOriCoef": vox_ori_resid_ori_coef,
            "voxPredOriR2": vox_pred_ori_r2,
            "voxResidOriR2": vox_resid_ori_r2,
            "voxOriPredOriR2": vox_ori_pred_ori_r2,
            "voxOriResidOriR2": vox_ori_resid_ori_r2,
            "sessBetas": sess_betas,
            "sessStdBetas": sess_std_betas,
            "roiInd": roi_ind,
        }

        # Choose filename suffix based on z-score method
        zscore_str = {
            1: "_zscore",
            2: "_zeroMean",
            3: "_equalStd",
            4: "_zeroROImean",
        }.get(to_zscore, "")

        # Save to .mat
        save_path = os.path.join(
            save_folder,
            f"regressPrfSplit_session_v{visual_region}_sub{isub}{zscore_str}.mat",
        )

        sio.savemat(
            save_path,
            {
                "nsd": nsd,
                "numLevels": num_levels,
                "numOrientations": num_orientations,
                "rois": rois,
                "nvox": nvox,
                "roiPrf": roi_prf,
                "nsplits": nsplits,
            },
        )

        print(f"Saved results to: {save_path}")


# Run on subject 1, only visual region 1 (V1v+d), no z-scoring, and DEBUG_MODE on
regress_prf_split_expand(isub=1, visual_regions=[1], to_zscore=0, DEBUG_MODE=True)

ModuleNotFoundError: No module named 'nibabel'

In [2]:
import h5py
import numpy as np

file_path = r"C:\nsd\prfsample_expand\prfSampleStim_v1_sub1.h5"

with h5py.File(file_path, "r") as f:
    # Scalars from attributes
    num_levels = int(f.attrs["numLevels"])
    num_orientations = int(f.attrs["numOrientations"])
    print(
        f"\nAttributes → num_levels: {num_levels}, num_orientations: {num_orientations}"
    )

    # Top-level arrays
    all_imgs = np.array(f["allImgs"])
    rois = np.array(f["rois"])
    print(f"Loaded allImgs: shape = {all_imgs.shape}, dtype = {all_imgs.dtype}")
    print(f"Loaded rois: shape = {rois.shape}, dtype = {rois.dtype}")

    # PRF data containers
    prf_sample_lev = {}
    prf_sample_lev_ori = {}

    print("\nLoading PRF samples:")
    for roi_id in rois:
        roi_key = f"roi_{int(roi_id)}"

        prf_lev = np.array(f[f"prfSampleLev/{roi_key}"])
        prf_ori = np.array(f[f"prfSampleLevOri/{roi_key}"])

        prf_sample_lev[int(roi_id)] = prf_lev
        prf_sample_lev_ori[int(roi_id)] = prf_ori

        print(f"\nROI {roi_id}:")
        print(f"  prfSampleLev     shape = {prf_lev.shape}, dtype = {prf_lev.dtype}")
        print(f"  prfSampleLevOri  shape = {prf_ori.shape}, dtype = {prf_ori.dtype}")


Attributes → num_levels: 7, num_orientations: 8
Loaded allImgs: shape = (975,), dtype = int32
Loaded rois: shape = (2,), dtype = int32

Loading PRF samples:

ROI 0:
  prfSampleLev     shape = (10000, 594, 9), dtype = float64
  prfSampleLevOri  shape = (10000, 594, 9, 8), dtype = float64

ROI 1:
  prfSampleLev     shape = (10000, 756, 9), dtype = float64
  prfSampleLevOri  shape = (10000, 756, 9, 8), dtype = float64


# Optimized

In [ ]:
import numpy as np
import nibabel as nib
import scipy.io as sio
import os
import h5py
from scipy.stats import zscore
from sklearn.linear_model import LinearRegression
from joblib import Parallel, delayed


# --------------------------------------------------
# Utility: Compute R² between residual and original
# --------------------------------------------------


def rsquared(Xresid, Xorig):
    return 1 - np.sum((Xresid) ** 2) / np.sum((Xorig - np.mean(Xorig)) ** 2)


# ----------------------------------------------------------------------------
# Main Function: Perform voxel-wise regression of PRF features on NSD betas
# isub: subject index (1-based like MATLAB)
# visual_regions: list of visual regions (e.g., [1, 2, 3])
# to_zscore: normalization method (0: none, 1: z-score, 2: mean, etc.)
# ----------------------------------------------------------------------------


def regress_prf_split_expand(isub, visual_regions=[1], to_zscore=0):
    # -------------------------------
    # Setup & File Paths
    # -------------------------------

    # Number of sessions per subject
    nsessions_sub = [40, 40, 32, 30, 40, 32, 40, 30]
    nsessions = nsessions_sub[isub - 1]
    nsplits = nsessions  # One split per session

    # bandpass info
    bandpass = 1
    band_min = 1
    band_max = 7

    # Set data folders
    prf_folder = r"C:\nsd\prfsample_expand"
    save_folder = r"C:\nsd\repDrift_expand/"
    nsd_folder = r"D:\NSD"
    betas_folder = os.path.join(nsd_folder, f"subj{isub:02d}", "func1pt8mm")
    visual_rois_file = os.path.join(betas_folder, "roi", "prf-visualrois.nii.gz")

    # Load visual ROI mask (NIfTI)
    vis_roi_img = nib.load(visual_rois_file)
    vis_roi_data = vis_roi_img.get_fdata().flatten()  # Flatten for voxel indexing

    roi_names = ["V1v", "V1d", "V2v", "V2d", "V3v", "V3d", "hV4"]

    # ------------------------------------------------------------
    # Define visual region mapping (visualRegion → ROI indices)
    # ------------------------------------------------------------

    # Each visualRegion value corresponds to a group of ROIs:
    #   1 → [V1v, V1d] (ROI indices 0 and 1)
    #   2 → [V2v, V2d] (ROI indices 2 and 3)
    #   3 → [V3v, V3d] (ROI indices 4 and 5)
    #   4 → [hV4]      (ROI index 6)
    roi_map = {
        1: [0, 1],  # V1v + V1d
        2: [2, 3],  # V2v + V2d
        3: [4, 5],  # V3v + V3d
        4: [6],  # hV4
    }

    # Load NSD experiment design
    nsd_design = sio.loadmat(os.path.join(nsd_folder, "nsd_expdesign.mat"))
    master_ordering = nsd_design["masterordering"].squeeze() - 1  # MATLAB 1-indexed
    sub_design = nsd_design["subjectim"][isub - 1, master_ordering].squeeze()

    # ------------------------------------------
    # Split trial matrix (1 row per split)
    # ------------------------------------------

    ntrials = len(master_ordering)
    trials_per_session = 10 * 75  # Each session: 10 blocks × 75 trials
    split_img_trials = np.zeros((nsplits, ntrials), dtype=bool)

    itrial = 0
    for isplit in range(nsplits):  # Define split_img_trials once
        split_img_trials[isplit, itrial : itrial + trials_per_session] = True
        itrial += trials_per_session

    # ------------------------------------------
    # Loop over Visual Regions
    # ------------------------------------------

    for (
        visual_region
    ) in visual_regions:  # # Outer loop: per visual ROI (e.g., V1v, V1d...)
        rois = roi_map.get(visual_region, [])
        print(f"Processing visual region: {visual_region}")

        # --- Load PRF sample features for this region ---
        prf_file = os.path.join(
            prf_folder,
            f"prfSampleStim_v{visual_region}_sub{isub}.h5",  # Load the PRF energy for this region
        )
        with h5py.File(prf_file, "r") as f:
            # Load scalar attributes
            num_levels = int(f.attrs["numLevels"])
            num_orientations = int(f.attrs["numOrientations"])
            roi_prf = None  # Optional — only used for saving, not needed for regression

            # Load global image list and ROI list
            all_imgs = np.array(f["allImgs"]).squeeze()
            rois = np.array(f["rois"]).squeeze()

            # Load prfSampleLev and prfSampleLevOri into Python dictionaries
            prf_sample_lev = {}
            prf_sample_lev_ori = {}

            for roinum in rois:
                prf_sample_lev[roinum] = np.array(
                    f[f"prfSampleLev/roi_{roinum}"]
                )  # shape: (n_imgs, n_vox, num_levels + 2)
                prf_sample_lev_ori[roinum] = np.array(
                    f[f"prfSampleLevOri/roi_{roinum}"]
                )  # shape: (n_imgs, n_vox, num_levels + 2, num_orientations)

        # --- Init containers for each ROI ---
        roi_betas = {roinum: [] for roinum in range(len(rois))}  # beta responses
        roi_ind = {}  # voxel indices per ROI

        # --- Load and normalize beta data per session ---
        for isession in range(1, nsessions + 1):  # For each session
            beta_file = os.path.join(betas_folder, f"betas_session{isession:02d}.nii")
            beta_img = nib.load(beta_file)
            beta_data = beta_img.get_fdata().astype(np.float64)
            beta_data = beta_data / 300  # Convert to percent signal change
            beta_data = beta_data.reshape(-1, beta_data.shape[-1])  # Flatten 4D to 2D

            for roinum, iroi in enumerate(
                rois
            ):  # For each ROI within this visual region
                vox_idx = vis_roi_data == iroi
                roi_ind[roinum] = np.where(vox_idx)[0]
                roi_voxels = beta_data[vox_idx, :]  # Get betas for this ROI

                # --- Apply normalization method ---
                if to_zscore == 1:
                    norm_voxels = zscore(roi_voxels, axis=1, ddof=0)
                elif to_zscore == 2:
                    norm_voxels = roi_voxels - roi_voxels.mean(axis=1, keepdims=True)
                elif to_zscore == 3:
                    mean_ = roi_voxels.mean(axis=1, keepdims=True)
                    norm_voxels = mean_ + zscore(roi_voxels - mean_, axis=1, ddof=0)
                elif to_zscore == 4:
                    norm_voxels = roi_voxels - roi_voxels.mean()
                else:
                    norm_voxels = roi_voxels  # No normalization

                roi_betas[roinum].append(norm_voxels)

        # --- Concatenate sessions along time axis ---
        for roinum in roi_betas:
            roi_betas[roinum] = np.concatenate(roi_betas[roinum], axis=1)

        # --- Match presented images to full image set ---
        img_trials_mask, img_num = (
            np.isin(sub_design, all_imgs, assume_unique=True),
            None,
        )
        img_num = np.searchsorted(
            all_imgs, sub_design[img_trials_mask]
        )  # Shape: (n_trials,)

        # --- Ensure trial counts match for this subject ---
        for roinum in roi_betas:
            roi_betas[roinum] = roi_betas[roinum][:, : split_img_trials.shape[1]]
        split_img_trials = split_img_trials[:, : roi_betas[roinum].shape[1]]
        img_num = img_num[: roi_betas[roinum].shape[1]]

        max_num_trials = split_img_trials.sum(axis=1).max()  # Used for array size later

        # Prepare containers per ROI
        nsd = {}
        nvox = {}
        r2 = {}
        r2ori = {}
        r2split = {}
        r2ori_split = {}
        rss_split = {}
        rss_ori_split = {}
        pearson_r = {}
        pearson_rori = {}
        sess_betas = {}
        sess_std_betas = {}

        vox_coef = {}
        vox_ori_coef = {}
        vox_pred_ori_coef = {}
        vox_ori_pred_ori_coef = {}
        vox_resid_ori_coef = {}
        vox_ori_resid_ori_coef = {}

        vox_pred_ori_r2 = {}
        vox_resid_ori_r2 = {}
        vox_ori_pred_ori_r2 = {}
        vox_ori_resid_ori_r2 = {}

        vox_coef_corr = {}
        vox_ori_coef_corr = {}
        vox_coef_corr_with_const = {}
        vox_ori_coef_corr_with_const = {}

        # ------------------------------------------------------------
        # Voxel-wise Regression for Each ROI and Each Split
        # ------------------------------------------------------------
        for roinum in range(len(rois)):
            print(f"  → Processing ROI {roinum} (ID {rois[roinum]})")
            nvox[roinum] = roi_betas[roinum].shape[0]
            L = nvox[roinum]

            # Allocate arrays for this ROI
            shape_lev = num_levels + 1
            shape_ori = num_levels * num_orientations + 1

            # Initialize coefficient and residual matrices per voxel per split
            vox_coef[roinum] = np.zeros((nsplits, L, shape_lev))
            vox_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))
            vox_pred_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))
            vox_ori_pred_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))
            vox_resid_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))
            vox_ori_resid_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))

            vox_pred_ori_r2[roinum] = np.zeros((nsplits, L))
            vox_resid_ori_r2[roinum] = np.zeros((nsplits, L))
            vox_ori_pred_ori_r2[roinum] = np.zeros((nsplits, L))
            vox_ori_resid_ori_r2[roinum] = np.zeros((nsplits, L))
            r2[roinum] = np.zeros((nsplits, L))
            r2ori[roinum] = np.zeros((nsplits, L))
            sess_betas[roinum] = np.zeros((nsplits, L))
            sess_std_betas[roinum] = np.zeros((nsplits, L))

            # Loop over splits
            for isplit in range(nsplits):
                img_trials_split = split_img_trials[isplit]
                trial_idx = np.where(img_trials_split)[0]

                betas_split = roi_betas[roinum][:, trial_idx]  # shape: voxels × trials
                sess_betas[roinum][isplit] = np.mean(betas_split, axis=1)
                sess_std_betas[roinum][isplit] = np.std(betas_split, axis=1)

                # ---------------------------
                # Precompute X matrix for all voxels
                # ---------------------------
                X_all_vox = prf_sample_lev[roinum][img_num[trial_idx], :, :]  # (n_trials, n_voxels, num_levels+2)
                n_trials, n_voxels, n_feats = X_all_vox.shape

                # Precompute design matrices for all voxels
                X_ori_all = prf_sample_lev_ori[roinum][img_num[trial_idx], :, :, :]  # (n_trials, n_voxels, levels, orientations)
                X_ori_flat = X_ori_all.reshape(X_ori_all.shape[0], X_ori_all.shape[1], -1)  # (n_trials, n_voxels, lev*ori)

                # Get extra features: lowest SF and highest SF
                X_basic = prf_sample_lev[roinum][img_num[trial_idx], :, :]  # (n_trials, n_voxels, num_levels+2)

                n_trials, n_voxels, _ = X_basic.shape
                n_features_ori = X_ori_flat.shape[-1] + 2 + 1  # +2 for lowest, highest SF, +1 for constant

                # Prebuild full design matrix: (voxels, trials, features)
                X_ori_stack = np.zeros((n_voxels, n_trials, n_features_ori))
                for ivox in range(n_voxels):
                    X_ori_stack[ivox, :, :-3] = X_ori_flat[:, ivox, :]
                    X_ori_stack[ivox, :, -3] = X_basic[:, ivox, -2]  # lowest SF
                    X_ori_stack[ivox, :, -2] = X_basic[:, ivox, -3]  # highest SF
                    X_ori_stack[ivox, :, -1] = 1  # constant

                # Flatten design matrix to shape (n_trials, n_voxels, n_feats)
                X_stack = np.zeros((n_trials, n_voxels, n_feats + 1))  # +1 for constant
                X_stack[:, :, :-1] = X_all_vox
                X_stack[:, :, -1] = 1  # Add constant column

                # Reshape to (n_trials, n_voxels * (features + 1))
                X_stack = X_stack.transpose(1, 0, 2)  # (n_voxels, n_trials, features+1)

                # ---------------------------
                # Solve all voxels at once
                # ---------------------------
                for ivox in range(L):
                    X = X_stack[ivox]  # (n_trials, features+1)
                    y = betas_split[ivox, :]  # (n_trials,)

                    # Solve least squares
                    coef = np.linalg.lstsq(X, y, rcond=None)[0]
                    vox_coef[roinum][isplit, ivox, :] = coef

                    pred = X @ coef
                    residual = y - pred
                    r2[roinum][isplit, ivox] = rsquared(residual, y)

                    # ---------- PRF: Orientation-Level Model ----------
                    X = X_ori_stack[ivox, :, :]  # (n_trials, features)
                    y = betas_split[ivox, :]

                    coef_ori = np.linalg.lstsq(X, y, rcond=None)[0]
                    vox_ori_coef[roinum][isplit, ivox, :] = coef_ori

                    pred_ori = X @ coef_ori
                    residual_ori = y - pred_ori
                    r2ori[roinum][isplit, ivox] = rsquared(residual_ori, y)

                    # ---------- Residual regressions ----------
                    # Predict orientation residuals with orientation model
                    vox_ori_resid_ori_coef[roinum][isplit, ivox, :] = np.linalg.lstsq(
                        X_ori, residual_ori, rcond=None
                    )[0]
                    pred_ori_resid = np.dot(
                        X_ori, vox_ori_resid_ori_coef[roinum][isplit, ivox, :]
                    )  # pred_ori_resid = X_ori @ vox_ori_resid_ori_coef[roinum][isplit, ivox, :]
                    vox_ori_resid_ori_r2[roinum][isplit, ivox] = rsquared(
                        residual_ori - pred_ori_resid, y
                    )

                    # Predict basic residuals with orientation model
                    vox_resid_ori_coef[roinum][isplit, ivox, :] = np.linalg.lstsq(
                        X_ori, residual, rcond=None
                    )[0]
                    pred_resid_ori = np.dot(
                        X_ori, vox_resid_ori_coef[roinum][isplit, ivox, :]
                    )  # pred_resid_ori = X_ori @ vox_resid_ori_coef[roinum][isplit, ivox, :]
                    vox_resid_ori_r2[roinum][isplit, ivox] = rsquared(
                        residual - pred_resid_ori, y
                    )

                    # Predict prediction with orientation model
                    pred_from_basic = pred
                    vox_pred_ori_coef[roinum][isplit, ivox, :] = np.linalg.lstsq(
                        X_ori, pred_from_basic, rcond=None
                    )[0]
                    pred_pred_ori = np.dot(
                        X_ori, vox_pred_ori_coef[roinum][isplit, ivox, :]
                    )
                    vox_pred_ori_r2[roinum][isplit, ivox] = rsquared(
                        pred_from_basic - pred_pred_ori, y
                    )

                    # Predict full orientation prediction with same
                    vox_ori_pred_ori_coef[roinum][isplit, ivox, :] = np.linalg.lstsq(
                        X_ori, pred_ori, rcond=None
                    )[0]
                    pred_ori_pred_ori = np.dot(
                        X_ori, vox_ori_pred_ori_coef[roinum][isplit, ivox, :]
                    )
                    vox_ori_pred_ori_r2[roinum][isplit, ivox] = rsquared(
                        pred_ori - pred_ori_pred_ori, y
                    )
            # Containers for cross-split comparisons
            r2split[roinum] = np.zeros((nsplits, L, nsplits))
            r2ori_split[roinum] = np.zeros((nsplits, L, nsplits))
            rss_split[roinum] = np.zeros((nsplits, L, nsplits))
            rss_ori_split[roinum] = np.zeros((nsplits, L, nsplits))
            pearson_r[roinum] = np.zeros((nsplits, L, nsplits))
            pearson_rori[roinum] = np.zeros((nsplits, L, nsplits))

            # ----------------------------
            # Loop: isplit (target session)
            # ----------------------------
            for isplit in range(nsplits):
                trial_idx = np.where(split_img_trials[isplit])[0]
                y = roi_betas[roinum][:, trial_idx]  # shape: (voxels, trials)

                for ivox in range(L):
                    y_vox = y[ivox, :]

                    # Get design matrices for isplit
                    X = prf_sample_lev[roinum][
                        img_num[trial_idx], :, :
                    ]  # shape: (n_trials, n_vox, num_levels+2)
                    X_vox = X[:, ivox, :]  # shape: (n_trials, num_levels+2)
                    X = np.column_stack([X_vox, np.ones(X_vox.shape[0])])

                    X_ori_all = prf_sample_lev_ori[roinum][
                        img_num[trial_idx], :, :, :
                    ]  # (n_trials, n_vox, lev, ori)
                    X_ori_vox = X_ori_all[:, ivox, :, :].reshape(
                        len(trial_idx), -1
                    )  # → (n_trials, lev * ori)
                    X_ori = np.column_stack(
                        [
                            X_ori_vox,
                            X_vox[:, -2],  # lowest SF
                            X_vox[:, -3],  # highest SF
                            np.ones(X_vox.shape[0]),
                        ]
                    )

                    # ----------------------------
                    # Loop: nextSplit (model source)
                    # ----------------------------
                    for next_split in range(nsplits):
                        coef = vox_coef[roinum][next_split, ivox, :]
                        coef_ori = vox_ori_coef[roinum][next_split, ivox, :]

                        # Predict from basic model
                        y_pred = np.dot(X, coef)  # y_pred = X @ coef
                        residual = y_vox - y_pred
                        r2split[roinum][isplit, ivox, next_split] = rsquared(
                            residual, y_vox
                        )
                        rss_split[roinum][isplit, ivox, next_split] = np.sum(
                            residual**2
                        )
                        pearson_r[roinum][isplit, ivox, next_split] = np.corrcoef(
                            y_vox, y_pred
                        )[0, 1]

                        # Predict from orientation model
                        y_pred_ori = np.dot(
                            X_ori, coef_ori
                        )  # y_pred_ori = X_ori @ coef_ori
                        residual_ori = y_vox - y_pred_ori
                        r2ori_split[roinum][isplit, ivox, next_split] = rsquared(
                            residual_ori, y_vox
                        )
                        rss_ori_split[roinum][isplit, ivox, next_split] = np.sum(
                            residual_ori**2
                        )
                        pearson_rori[roinum][isplit, ivox, next_split] = np.corrcoef(
                            y_vox, y_pred_ori
                        )[0, 1]

            # --------------------------------------------------------
            # Correlation Between Splits for Each Voxel
            # --------------------------------------------------------
            vox_coef_corr[roinum] = np.zeros((L, nsplits, nsplits))
            vox_ori_coef_corr[roinum] = np.zeros((L, nsplits, nsplits))
            vox_coef_corr_with_const[roinum] = np.zeros((L, nsplits, nsplits))
            vox_ori_coef_corr_with_const[roinum] = np.zeros((L, nsplits, nsplits))

            for ivox in range(L):
                # Without constant (drop last column)
                coef_mat = vox_coef[roinum][:, ivox, :-1]
                ori_mat = vox_ori_coef[roinum][:, ivox, :-1]

                vox_coef_corr[roinum][ivox] = np.corrcoef(coef_mat, rowvar=False)
                vox_ori_coef_corr[roinum][ivox] = np.corrcoef(ori_mat, rowvar=False)

                # With constant
                coef_mat_full = vox_coef[roinum][:, ivox, :]
                ori_mat_full = vox_ori_coef[roinum][:, ivox, :]

                vox_coef_corr_with_const[roinum][ivox] = np.corrcoef(
                    coef_mat_full, rowvar=False
                )
                vox_ori_coef_corr_with_const[roinum][ivox] = np.corrcoef(
                    ori_mat_full, rowvar=False
                )

        # -----------------------------------------------------
        # Package Results in Dictionary and Save
        # -----------------------------------------------------
        nsd = {
            "voxOriCoefCorrWithConst": vox_ori_coef_corr_with_const,
            "voxCoefCorrWithConst": vox_coef_corr_with_const,
            "voxOriCoefCorr": vox_ori_coef_corr,
            "voxCoefCorr": vox_coef_corr,
            "r2": r2,
            "r2ori": r2ori,
            "r2split": r2split,
            "r2oriSplit": r2ori_split,
            "rssOriSplit": rss_ori_split,
            "rssSplit": rss_split,
            "pearsonRori": pearson_rori,
            "pearsonR": pearson_r,
            "imgNum": img_num,
            "splitImgTrials": split_img_trials,
            "voxCoef": vox_coef,
            "voxOriCoef": vox_ori_coef,
            "voxPredOriCoef": vox_pred_ori_coef,
            "voxOriPredOriCoef": vox_ori_pred_ori_coef,
            "voxResidOriCoef": vox_resid_ori_coef,
            "voxOriResidOriCoef": vox_ori_resid_ori_coef,
            "voxPredOriR2": vox_pred_ori_r2,
            "voxResidOriR2": vox_resid_ori_r2,
            "voxOriPredOriR2": vox_ori_pred_ori_r2,
            "voxOriResidOriR2": vox_ori_resid_ori_r2,
            "sessBetas": sess_betas,
            "sessStdBetas": sess_std_betas,
            "roiInd": roi_ind,
        }

        # Choose filename suffix based on z-score method
        zscore_str = {
            1: "_zscore",
            2: "_zeroMean",
            3: "_equalStd",
            4: "_zeroROImean",
        }.get(to_zscore, "")

        # Save to .mat
        save_path = os.path.join(
            save_folder,
            f"regressPrfSplit_session_v{visual_region}_sub{isub}{zscore_str}.mat",
        )

        sio.savemat(
            save_path,
            {
                "nsd": nsd,
                "numLevels": num_levels,
                "numOrientations": num_orientations,
                "rois": rois,
                "nvox": nvox,
                "roiPrf": roi_prf,
                "nsplits": nsplits,
            },
        )

        print(f"Saved results to: {save_path}")